In [1]:
import os
import pathlib
import matplotlib.pyplot as plt
import json
from model_functions import test
import torch
from medmnist import ChestMNIST
from torchvision.transforms import v2
from model import ResNet18

node_logs = []

nodes = 10

tf = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32,scale=True)
])

test_data = ChestMNIST(split='test',transform=tf,download=True,root="./data")
val_data = ChestMNIST(split='val',transform=tf,download=True,root="./data")

classes = len(test_data.info["label"])
channels = test_data.info["n_channels"]
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ResNet18(channels,classes).to(device)


In [2]:
local_test_results = []

test_dl = torch.utils.data.DataLoader(test_data,16384)

for i in range(nodes):
    node = pathlib.Path(f"./results_sim/node_{i}").resolve()
    model.load_state_dict(torch.load(node/"best_model.pth"))
    if node.exists():
        print(f"Node {i}")
        best_model = node.glob("*.pth")
        _,test_acc = test(test_dl,model,loss_fn=torch.nn.BCEWithLogitsLoss(),device=device)
        local_test_results.append((f"node_{i}",test_acc))


Node 0
Test Error: Accuracy: 94.7%, Average Loss: 0.180238
Node 1
Test Error: Accuracy: 94.7%, Average Loss: 0.181786
Node 2
Test Error: Accuracy: 94.7%, Average Loss: 0.193489
Node 3
Test Error: Accuracy: 94.7%, Average Loss: 0.185340
Node 4
Test Error: Accuracy: 94.8%, Average Loss: 0.177662
Node 5
Test Error: Accuracy: 94.7%, Average Loss: 0.182327
Node 6
Test Error: Accuracy: 94.7%, Average Loss: 0.177970
Node 7
Test Error: Accuracy: 94.7%, Average Loss: 0.182649
Node 8
Test Error: Accuracy: 94.7%, Average Loss: 0.200400
Node 9
Test Error: Accuracy: 94.7%, Average Loss: 0.176781


In [3]:
global_models_results = []

global_model_dir = pathlib.Path(f"./results_sim/global_models").resolve()

best_val_global_model = None
best_val_acc = -1

val_dl = torch.utils.data.DataLoader(val_data,16384)
test_dl = torch.utils.data.DataLoader(test_data,16384)

models = global_model_dir.glob("*.pth")
for gm in models:
    print(gm)
    model.load_state_dict(torch.load(gm))
    _,val_acc = test(test_dl,model,loss_fn=torch.nn.BCEWithLogitsLoss(),device=device)
    if val_acc > best_val_acc:
        best_val_global_model = gm
        best_val_acc = val_acc
    
if best_val_global_model != None:
    model.load_state_dict(torch.load(best_val_global_model))
    _,test_acc = test(test_dl,model,loss_fn=torch.nn.BCEWithLogitsLoss(),device=device)
    global_models_results.append((best_val_global_model.name,test_acc))

/mnt/sdc/Hasan/Documents/My Documents/Code/GitLab/kademlia-based-federated-learning/results_sim/global_models/global_model_1778543845.3306708.pth
Test Error: Accuracy: 94.7%, Average Loss: 0.197616
/mnt/sdc/Hasan/Documents/My Documents/Code/GitLab/kademlia-based-federated-learning/results_sim/global_models/global_model_1778544377.5881422.pth
Test Error: Accuracy: 94.7%, Average Loss: 0.177512
/mnt/sdc/Hasan/Documents/My Documents/Code/GitLab/kademlia-based-federated-learning/results_sim/global_models/global_model_1778544859.8765345.pth
Test Error: Accuracy: 94.7%, Average Loss: 0.176210
/mnt/sdc/Hasan/Documents/My Documents/Code/GitLab/kademlia-based-federated-learning/results_sim/global_models/global_model_1778545330.538672.pth
Test Error: Accuracy: 94.7%, Average Loss: 0.176476
/mnt/sdc/Hasan/Documents/My Documents/Code/GitLab/kademlia-based-federated-learning/results_sim/global_models/global_model_1778545797.9609094.pth
Test Error: Accuracy: 94.7%, Average Loss: 0.177585
/mnt/sdc/Ha

In [4]:
print(local_test_results)
print(global_models_results)

[('node_0', 0.9474466825021334), ('node_1', 0.9474243939094356), ('node_2', 0.9474180257400932), ('node_3', 0.9474180257400932), ('node_4', 0.9475071801108857), ('node_5', 0.9474371302481203), ('node_6', 0.9473065827766013), ('node_7', 0.9473798167240379), ('node_8', 0.9474339461634493), ('node_9', 0.9472237965751507)]
[('global_model_1778544377.5881422.pth', 0.9474626029254892)]


In [5]:
import pandas as pd

pd.DataFrame(local_test_results)

,0,1
0,node_0,0.947447
1,node_1,0.947424
2,node_2,0.947418
3,node_3,0.947418
4,node_4,0.947507
5,node_5,0.947437
6,node_6,0.947307
7,node_7,0.947380
8,node_8,0.947434
9,node_9,0.947224


In [6]:
from numpy import average

node_results = [node[1] for node in local_test_results]
best_node = max(node_results)
average_node = average(node_results)
worst_node = min(node_results)

print(f"Best Node Accuracy : {best_node}")
print(f"Average Node Accuracy : {average_node}")
print(f"Worst Node Accuracy : {worst_node}")

Best Node Accuracy : 0.9475071801108857
Average Node Accuracy : 0.947399558049
Worst Node Accuracy : 0.9472237965751507


In [7]:
import pandas as pd

# the test result from the best global model from validation accuracy testing

pd.DataFrame(global_models_results)

,0,1
0,global_model_1778544377.5881422.pth,0.947463
